# HITL用 血管セグメンテーションマスク生成

U-Net (Dice=0.621, 4-fold CV) を multicenter study の top10 画像 (347 ケース, 3,089 枚) に適用し、
HITL (Human-In-The-Loop) アノテーション修正の起点となる予測マスクを生成する。

## Pipeline (1画像あたり)
1. フル画像読み込み (selected_images_disc_retina)
2. RT-DETR lens 検出 → bbox クロップ
3. 円形マスク適用 (学習時と同じ前処理)
4. 512×512 にリサイズ
5. ImageNet 正規化 → U-Net 4-fold ensemble → sigmoid 平均 → 0.5 閾値
6. 円形マスク外を除外 → RGBA PNG マスク生成

## 出力
```
E:\Kaisho_vascular_annotation\Before_HITL\
  images/    # 3,089枚 lens crop + 円形マスク済み画像 (512×512)
  masks/     # 3,089枚 RGBA PNG マスク (512×512, Red=血管, White=背景)
```

In [9]:
import cv2
import numpy as np
import torch
import pandas as pd
from pathlib import Path
from typing import Optional, Tuple
from tqdm import tqdm
import segmentation_models_pytorch as smp
from ultralytics import RTDETR
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

# === Paths ===
PROJECT_ROOT = Path(r'C:\Users\ykita\ROP_AI_project\ROP_project')
VASCULAR_DIR = PROJECT_ROOT / 'vascular_segmentation'

# Input
TOP10_CSV = PROJECT_ROOT / 'multicenter_study' / 'outputs_clinical_v3' / 'top10_selected_images.csv'
IMAGE_SRC_DIR = Path(r'E:\Multicenter_ROP_study\Multicenter_images\selected_images_disc_retina')

# Models
RTDETR_MODEL_PATH = PROJECT_ROOT / 'models' / 'rtdetr-l-1697_1703.pt'
UNET_WEIGHTS_DIR = VASCULAR_DIR / 'outputs'
N_FOLDS = 4
TARGET_SIZE = 512

# Output
OUTPUT_ROOT = Path(r'E:\Kaisho_vascular_annotation\Before_HITL')
OUTPUT_IMG_DIR = OUTPUT_ROOT / 'images'
OUTPUT_MASK_DIR = OUTPUT_ROOT / 'masks'

for d in [OUTPUT_IMG_DIR, OUTPUT_MASK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Quadro RTX 5000


## 1. Model Loading

In [10]:
# --- Lens detection helpers (from infer_case_video.py) ---

def pick_best_lens_bbox(det_results) -> Optional[np.ndarray]:
    """RT-DETR結果から Lens(cls=0) bbox(xyxy) を1つ選ぶ（conf最大を優先）。"""
    best = None
    best_conf = -1.0
    for r in det_results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        boxes = r.boxes
        cls_ids = boxes.cls.detach().cpu().numpy().astype(int)
        confs = boxes.conf.detach().cpu().numpy() if boxes.conf is not None else None
        xyxy = boxes.xyxy.detach().cpu().numpy()
        for i in range(len(cls_ids)):
            if int(cls_ids[i]) != 0:
                continue
            conf = float(confs[i]) if confs is not None else 0.0
            if conf > best_conf:
                best_conf = conf
                best = xyxy[i]
    return best


def clamp_xyxy(xyxy: np.ndarray, w: int, h: int) -> Tuple[int, int, int, int]:
    x1, y1, x2, y2 = [int(c) for c in xyxy]
    x1 = max(0, min(x1, w - 1))
    y1 = max(0, min(y1, h - 1))
    x2 = max(0, min(x2, w))
    y2 = max(0, min(y2, h))
    if x2 <= x1:
        x2 = min(w, x1 + 1)
    if y2 <= y1:
        y2 = min(h, y1 + 1)
    return x1, y1, x2, y2


def apply_circular_mask_to_crop(crop_bgr: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    """円形マスク（クロップ内中心・直径=(w+h)/2）を適用。"""
    h, w = crop_bgr.shape[:2]
    center_x, center_y = w // 2, h // 2
    diameter = (w + h) / 2.0
    radius = int(diameter / 2.0)
    circle_mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(circle_mask, (center_x, center_y), radius, 255, -1)
    masked = crop_bgr.copy()
    masked[circle_mask == 0] = (114, 114, 114)
    return masked, circle_mask


# Validation transform (ImageNet normalization)
val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

print('Helper functions defined')

Helper functions defined


In [11]:
# Load RT-DETR
rtdetr = RTDETR(str(RTDETR_MODEL_PATH))
print(f'RT-DETR loaded: {RTDETR_MODEL_PATH.name}')

# Load 4-fold U-Net ensemble
unet_models = []
for fold in range(1, N_FOLDS + 1):
    model = smp.Unet(
        encoder_name='resnet34',
        encoder_weights=None,
        in_channels=3,
        classes=1,
        activation=None
    ).to(device)
    weight_path = UNET_WEIGHTS_DIR / f'unet_fold{fold}.pt'
    model.load_state_dict(torch.load(str(weight_path), map_location=device, weights_only=True))
    model.eval()
    unet_models.append(model)
    print(f'  Fold {fold} loaded: {weight_path.name}')

print(f'\nModels ready: RT-DETR + {len(unet_models)}-fold U-Net ensemble')

RT-DETR loaded: rtdetr-l-1697_1703.pt
  Fold 1 loaded: unet_fold1.pt
  Fold 2 loaded: unet_fold2.pt
  Fold 3 loaded: unet_fold3.pt
  Fold 4 loaded: unet_fold4.pt

Models ready: RT-DETR + 4-fold U-Net ensemble


## 2. Image List

In [12]:
df_top10 = pd.read_csv(TOP10_CSV)
image_names = df_top10['image_name'].unique().tolist()
n_cases = df_top10['video_id'].nunique()

print(f'Total images: {len(image_names)}')
print(f'Total cases: {n_cases}')
print(f'Source dir: {IMAGE_SRC_DIR}')

# Verify all images exist
missing = [name for name in image_names if not (IMAGE_SRC_DIR / name).exists()]
if missing:
    print(f'WARNING: {len(missing)} images not found!')
    for m in missing[:10]:
        print(f'  {m}')
else:
    print(f'All {len(image_names)} images verified in source directory')

Total images: 3089
Total cases: 347
Source dir: E:\Multicenter_ROP_study\Multicenter_images\selected_images_disc_retina
All 3089 images verified in source directory


## 3. Inference Functions

In [13]:
@torch.no_grad()
def predict_ensemble(img_512_bgr: np.ndarray, circle_mask_512: np.ndarray) -> np.ndarray:
    """4-fold ensemble prediction on a 512x512 image.
    
    Returns:
        Binary mask (512x512, uint8, 0 or 255) with circle mask applied.
    """
    img_rgb = cv2.cvtColor(img_512_bgr, cv2.COLOR_BGR2RGB)
    transformed = val_transform(image=img_rgb)
    img_tensor = transformed['image'].unsqueeze(0).to(device)
    
    # Average sigmoid outputs across folds
    sigmoid_sum = torch.zeros(1, 1, TARGET_SIZE, TARGET_SIZE, device=device)
    for model in unet_models:
        logits = model(img_tensor)
        sigmoid_sum += torch.sigmoid(logits)
    sigmoid_avg = sigmoid_sum / len(unet_models)
    
    # Threshold + circle mask
    pred_512 = (sigmoid_avg > 0.5).squeeze().cpu().numpy().astype(np.uint8) * 255
    pred_512[circle_mask_512 == 0] = 0
    return pred_512


def binary_to_rgba(binary_mask: np.ndarray) -> np.ndarray:
    """Convert binary mask (0/255) to RGBA PNG format.
    
    Vessel (255) -> Red (R=255, G=0, B=0, A=255)
    Background (0) -> White (R=255, G=255, B=255, A=255)
    """
    h, w = binary_mask.shape[:2]
    rgba = np.full((h, w, 4), 255, dtype=np.uint8)  # White + full alpha
    vessel = binary_mask > 0
    rgba[vessel, 1] = 0  # G=0
    rgba[vessel, 2] = 0  # B=0 (RGBA order)
    return rgba


def process_single_image(img_path: Path) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    """Full pipeline: read -> lens detect -> crop -> circle mask -> resize -> U-Net -> mask.
    
    Returns:
        (img_512, pred_binary_512) or None on failure.
        img_512: 円形マスク適用済みの512x512 BGR画像
        pred_binary_512: 512x512 binary mask (0/255)
    """
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        return None
    img_h, img_w = img_bgr.shape[:2]
    
    # Lens detection
    det = rtdetr.predict(img_bgr, verbose=False)
    bbox = pick_best_lens_bbox(det)
    
    if bbox is None:
        x1, y1, x2, y2 = 0, 0, img_w, img_h
    else:
        x1, y1, x2, y2 = clamp_xyxy(bbox, img_w, img_h)
    
    # Crop + circular mask
    crop_bgr = img_bgr[y1:y2, x1:x2]
    masked_crop, circle_mask = apply_circular_mask_to_crop(crop_bgr)
    
    # Resize to 512x512
    img_512 = cv2.resize(masked_crop, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_LINEAR)
    circle_512 = cv2.resize(circle_mask, (TARGET_SIZE, TARGET_SIZE), interpolation=cv2.INTER_NEAREST)
    
    # U-Net ensemble prediction
    pred_512 = predict_ensemble(img_512, circle_512)
    
    return img_512, pred_512


print('Inference functions defined')

Inference functions defined


## 4. Batch Inference + Save

In [14]:
results_log = []
n_failed = 0
n_no_lens = 0

for img_name in tqdm(image_names, desc='Inference'):
    src_path = IMAGE_SRC_DIR / img_name
    result = process_single_image(src_path)
    
    if result is None:
        n_failed += 1
        print(f'FAILED to read: {img_name}')
        continue
    
    img_512, pred_binary = result
    
    # Save image (512x512, circular-masked)
    cv2.imwrite(str(OUTPUT_IMG_DIR / img_name), img_512)
    
    # Save mask (RGBA PNG)
    pred_rgba = binary_to_rgba(pred_binary)
    pred_bgra = cv2.cvtColor(pred_rgba, cv2.COLOR_RGBA2BGRA)
    cv2.imwrite(str(OUTPUT_MASK_DIR / img_name), pred_bgra)
    
    # Stats
    vessel_pixels = int(np.sum(pred_binary > 0))
    total_pixels = TARGET_SIZE * TARGET_SIZE
    
    results_log.append({
        'image_name': img_name,
        'vessel_pixels': vessel_pixels,
        'vessel_ratio': vessel_pixels / total_pixels,
    })

print(f'\nInference complete: {len(results_log)}/{len(image_names)} images')
print(f'Failed: {n_failed}')

Inference: 100%|██████████| 3089/3089 [09:11<00:00,  5.61it/s]


Inference complete: 3089/3089 images
Failed: 0


## 5. Verification

In [15]:
img_files = set(p.name for p in OUTPUT_IMG_DIR.glob('*.png'))
mask_files = set(p.name for p in OUTPUT_MASK_DIR.glob('*.png'))

print(f'Images: {len(img_files)} files')
print(f'Masks: {len(mask_files)} files')

imgs_without_masks = img_files - mask_files
masks_without_imgs = mask_files - img_files

if imgs_without_masks:
    print(f'WARNING: {len(imgs_without_masks)} images without masks')
if masks_without_imgs:
    print(f'WARNING: {len(masks_without_imgs)} masks without images')

# Check size and format (sample 20)
common = sorted(img_files & mask_files)
sample = common[:20]
issues = 0

for name in sample:
    img = cv2.imread(str(OUTPUT_IMG_DIR / name))
    mask = cv2.imread(str(OUTPUT_MASK_DIR / name), cv2.IMREAD_UNCHANGED)
    
    if img.shape[:2] != (TARGET_SIZE, TARGET_SIZE):
        print(f'IMG SIZE ERROR: {name} = {img.shape[:2]}')
        issues += 1
    
    if mask.shape != (TARGET_SIZE, TARGET_SIZE, 4):
        print(f'MASK FORMAT ERROR: {name} = {mask.shape}')
        issues += 1

if issues == 0:
    print(f'Format check passed (sampled {len(sample)} pairs): all 512x512, masks RGBA')

if len(img_files) == len(mask_files) == len(image_names) and not imgs_without_masks and not masks_without_imgs and issues == 0:
    print(f'\nAll checks passed: {len(img_files)} image-mask pairs')
else:
    print(f'\nWARNING: Verification issues detected')

Images: 3089 files
Masks: 3089 files
Format check passed (sampled 20 pairs): all 512x512, masks RGBA

All checks passed: 3089 image-mask pairs


## 6. QC Visualization

In [ ]:
np.random.seed(42)
common = sorted(img_files & mask_files)
qc_names = np.random.choice(common, size=min(10, len(common)), replace=False)

fig, axes = plt.subplots(len(qc_names), 2, figsize=(8, 3.5 * len(qc_names)))
if len(qc_names) == 1:
    axes = axes[np.newaxis, :]

for idx, name in enumerate(qc_names):
    img = cv2.imread(str(OUTPUT_IMG_DIR / name))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask_bgra = cv2.imread(str(OUTPUT_MASK_DIR / name), cv2.IMREAD_UNCHANGED)
    
    # Extract vessel from BGRA: vessel = B==0 & G==0
    vessel = (mask_bgra[:, :, 0] == 0) & (mask_bgra[:, :, 1] == 0)
    
    # Original image
    axes[idx, 0].imshow(img_rgb)
    axes[idx, 0].set_title(name, fontsize=7)
    axes[idx, 0].axis('off')
    
    # Overlay: green vessels on image
    overlay = img_rgb.copy()
    overlay[vessel] = [50, 255, 50]
    blended = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)
    
    vessel_pct = 100 * np.sum(vessel) / (TARGET_SIZE * TARGET_SIZE)
    axes[idx, 1].imshow(blended)
    axes[idx, 1].set_title(f'U-Net mask (vessel={vessel_pct:.2f}%)', fontsize=8)
    axes[idx, 1].axis('off')

plt.suptitle('QC: U-Net Vessel Segmentation (lens crop + circular mask)', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Summary

In [17]:
df_results = pd.DataFrame(results_log)

print('=== Inference Summary ===')
print(f'Total images processed: {len(df_results)}')
print(f'Output directory: {OUTPUT_ROOT}')
print(f'Image size: {TARGET_SIZE}x{TARGET_SIZE}')
print(f'Mask format: RGBA PNG (Red=vessel, White=background)')
print()
print(f'Vessel pixel ratio (512x512):')
print(f'  Mean:   {df_results["vessel_ratio"].mean():.4f}')
print(f'  Median: {df_results["vessel_ratio"].median():.4f}')
print(f'  Min:    {df_results["vessel_ratio"].min():.4f}')
print(f'  Max:    {df_results["vessel_ratio"].max():.4f}')
print()
print(f'Zero-vessel masks: {(df_results["vessel_ratio"] == 0).sum()}')

# Save results log
results_csv = VASCULAR_DIR / 'inference_multicenter_results.csv'
df_results.to_csv(results_csv, index=False)
print(f'\nResults log saved to: {results_csv}')

=== Inference Summary ===
Total images processed: 3089
Output directory: E:\Kaisho_vascular_annotation\Before_HITL
Image size: 512x512
Mask format: RGBA PNG (Red=vessel, White=background)

Vessel pixel ratio (512x512):
  Mean:   0.0656
  Median: 0.0650
  Min:    0.0026
  Max:    0.1526

Zero-vessel masks: 0

Results log saved to: C:\Users\ykita\ROP_AI_project\ROP_project\vascular_segmentation\inference_multicenter_results.csv
